# TUGAS MANDIRI

## Konteks / Skenario

"Kopi Nusantara" ingin membangun model untuk memprediksi apakah seorang pelanggan akan memberikan ulasan positif (rating ≥ 4) atau tidak, berdasarkan atribut pesanan mereka — berguna untuk mengidentifikasi kombinasi menu/cabang yang berisiko mengecewakan pelanggan sebelum benar-benar terjadi.

### Menyiapkan Dataset

In [2]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml import Pipeline

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas12-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_ulasan = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)

### A. Membangun ML Pipeline 

Buat Pipeline dengan tahapan: StringIndexer untuk nama_menu, cabang, waktu_pesan → OneHotEncoder untuk ketiganya → VectorAssembler (gabungkan hasil one-hot encoding dengan qty dan waktu_tunggu_menit) → LogisticRegression dengan labelCol="ulasan_positif".

In [6]:
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

ohe = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

assembler = VectorAssembler(
    inputCols=["qty", "waktu_tunggu_menit", "menu_vec", "cabang_vec", "waktu_vec"],
    outputCol="features"
)

lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])

### B. Pelatihan & Evaluasi 

Bagi data 80:20 (seed=77), latih pipeline, lalu evaluasi pada data testing menggunakan accuracy, F1-score, dan AUC-ROC. Tampilkan juga confusion matrix sederhana menggunakan groupBy.

In [7]:
train_data, test_data = df_ulasan.randomSplit([0.8, 0.2], seed=77)

model_lr = pipeline_lr.fit(train_data)
pred_lr = model_lr.transform(test_data)

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(pred_lr)
f1_lr = ev_f1.evaluate(pred_lr)
auc_lr = ev_auc.evaluate(pred_lr)

print("=== EVALUASI LOGISTIC REGRESSION (BAGIAN B) ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-Score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\nConfusion Matrix Logistic Regression:")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

=== EVALUASI LOGISTIC REGRESSION (BAGIAN B) ===
Accuracy : 0.7929
F1-Score : 0.7926
AUC-ROC  : 0.8650

Confusion Matrix Logistic Regression:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



### C. Model Pembanding 

Latih RandomForestClassifier (numTrees=30) menggunakan pipeline dengan lima tahap feature engineering yang identik dari bagian A (hanya tahap terakhir yang diganti). Bandingkan ketiga metrik evaluasinya dengan model Logistic Regression pada bagian B.

In [8]:
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30, seed=77)
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train_data)
pred_rf = model_rf.transform(test_data)

acc_rf = ev_acc.evaluate(pred_rf)
f1_rf = ev_f1.evaluate(pred_rf)
auc_rf = ev_auc.evaluate(pred_rf)

print("=== EVALUASI RANDOM FOREST (BAGIAN C) ===")
print(f"Accuracy : {acc_rf:.4f}")
print(f"F1-Score : {f1_rf:.4f}")
print(f"AUC-ROC  : {auc_rf:.4f}")


=== EVALUASI RANDOM FOREST (BAGIAN C) ===
Accuracy : 0.7946
F1-Score : 0.7943
AUC-ROC  : 0.8555


### D. Analisis Kesalahan Model 

Dari hasil prediksi model terbaik (pilih salah satu dari B atau C berdasarkan AUC tertinggi), tampilkan 10 baris di mana model salah memprediksi (label != prediction). Amati: apakah ada pola tertentu pada kesalahan tersebut (mis. mayoritas terjadi pada waktu_tunggu_menit yang berada di sekitar ambang batas tertentu)?

In [9]:
best_pred = pred_lr if auc_lr >= auc_rf else pred_rf

print("\n=== 10 BARIS PREDIKSI YANG SALAH (BAGIAN D) ===")
best_pred.filter(col("ulasan_positif") != col("prediction")) \
    .select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability") \
    .show(10, truncate=False)


=== 10 BARIS PREDIKSI YANG SALAH (BAGIAN D) ===
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414356,0.37461208245856437]|
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149019,0.08861774338509809]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008503,0.7679086206099149]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016906074,0.5738454598309393]|
|Americano|Magelang|Sore       |3  |5                 |0             |1.0       |[0.165340

### E. Kesimpulan Bisnis 

Tulis pada markdown cell (minimal 100 kata): berdasarkan model yang telah dibangun, apa rekomendasi konkret yang dapat diberikan kepada manajemen "Kopi Nusantara" untuk meningkatkan kepuasan pelanggan? Kaitkan dengan fitur yang paling berpengaruh terhadap prediksi (boleh merujuk pada pola yang ditemukan di bagian D).